# adpo_dataset

In [ ]:
import pandas as pd
import json
import os

knowledge_path = "../seimei_knowledge/exp4_train.csv"
seimei_run_dir = "../seimei_runs/"
dataset_save_path = "../exp4/adpo_dataset.json"
n_sample = 5

klg_df = pd.read_csv(knowledge_path)

dataset = []

for i in range(len(klg_df)):
    run_id = klg_df.iloc[i]["run_id"]
    knowledge = klg_df.iloc[i]["knowledge"]
    condition = klg_df.iloc[i]["condition"]

    positive_key = f"Condition: {condition}\nKnowledge: {knowledge}"

    # ---- load query from corresponding run ----
    msg_path = os.path.join(seimei_run_dir, run_id, "messages.json")
    with open(msg_path, "r") as f:
        messages = json.load(f)
    
    query = None
    for message_dict in messages:
        if message_dict.get("role") == "user":
            query = message_dict.get("content", "")
            break

    if query is None:
        # no user message found; skip this row
        continue

    # extract the actual query text (adjust this if your prompt format changes)
    split_token = "answer the question below:\n\n"
    if split_token in query:
        query = query.split(split_token, 1)[1]

    # ---- sample negatives: rows with different run_id ----
    candidates = klg_df[klg_df["run_id"] != run_id]

    # if dataset smaller than n_sample, sample as many as possible
    n_neg = min(n_sample, len(candidates))
    sampled_rows = candidates.sample(n=n_neg, replace=False, random_state=42)

    sample_keys = [
        f"Condition: {row['condition']}\nKnowledge: {row['knowledge']}"
        for _, row in sampled_rows.iterrows()
    ]

    # if for some reason < n_sample, you can choose to skip or pad; here we skip incomplete batches
    if len(sample_keys) < n_sample:
        continue

    # ---- build batch + dpo_pairs ----
    batch = [
        {
            "msg": [
                {
                    "role": "user",
                    "content": (
                        "Give me relevance score between query and key;\n\n"
                        f"<query>{query}</query>\n\n<key>{positive_key}</key>"
                    ),
                }
            ]
        }
    ]

    for k in sample_keys:
        batch.append({
            "msg": [
                {
                    "role": "user",
                    "content": (
                        "Give me relevance score between query and key;\n\n"
                        f"<query>{query}</query>\n\n<key>{k}</key>"
                    ),
                }
            ]
        })

    dpo_pairs = [[0, j] for j in range(1, n_sample + 1)]

    dataset.append({
        "batch": batch,
        "dpo_pairs": dpo_pairs,
    })

# ---- save ----
os.makedirs(os.path.dirname(dataset_save_path), exist_ok=True)
with open(dataset_save_path, "w") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)


# nDCG of knowledge

In [ ]:
"""
queries.jsonl :
```
{"_id": 0, "text":"..."}
{"_id": 1, "text":"..."}
...
```

corpus.jsonl : 
```
{"_id": 0, "text":"..."}
{"_id": 1, "text":"..."}
...
```

qrels/test.tsv : 
```
query-id	corpus-id	score
0	1	1
0	5	1
...
```
"""


import pandas as pd
import json
import os

knowledge_path = "../seimei_knowledge/exp4_train.csv"
seimei_run_dir = "../seimei_runs/"
query_save_path = "../exp4/train_klg_retrieval_eval/queries.jsonl"
corpus_save_path = "../exp4/train_klg_retrieval_eval/corpus.jsonl"
qrel_save_path = "../exp4/train_klg_retrieval_eval/qrels/test.tsv"

klg_df = pd.read_csv(knowledge_path)

for i in range(len(klg_df)):
    run_id = klg_df.iloc[i]["run_id"]
    knowledge = klg_df.iloc[i]["knowledge"]
    condition = klg_df.iloc[i]["condition"]

    positive_key = f"Condition: {condition}\nKnowledge: {knowledge}"

    # ---- load query from corresponding run ----
    msg_path = os.path.join(seimei_run_dir, run_id, "messages.json")
    with open(msg_path, "r") as f:
        messages = json.load(f)
    
    query = None
    for message_dict in messages:
        if message_dict.get("role") == "user":
            query = message_dict.get("content", "")
            break

    if query is None:
        # no user message found; skip this row
        continue

    # extract the actual query text (adjust this if your prompt format changes)
    split_token = "answer the question below:\n\n"
    if split_token in query:
        query = query.split(split_token, 1)[1]

    query_id = run_id
    corpus_id = i
    corpus = positive_key
    score = 1

    query_row = {"_id": query_id, "text": query}
    corpus_row = {"_id": corpus_id, "text": corpus}
    qrels_row = {"query-id": query_id, "corpus-id":corpus_id, "score": score}

    
# ---- save ----


In [3]:
import pandas as pd
import json
import os

knowledge_path = "../seimei_knowledge/exp4_train.csv"
seimei_run_dir = "../seimei_runs/"
query_save_path = "../exp4/train_klg_retrieval_eval/queries.jsonl"
corpus_save_path = "../exp4/train_klg_retrieval_eval/corpus.jsonl"
qrel_save_path = "../exp4/train_klg_retrieval_eval/qrels/test.tsv"

# ---- prepare output dirs ----
os.makedirs(os.path.dirname(query_save_path), exist_ok=True)
os.makedirs(os.path.dirname(corpus_save_path), exist_ok=True)
os.makedirs(os.path.dirname(qrel_save_path), exist_ok=True)

klg_df = pd.read_csv(knowledge_path)

# We'll deduplicate queries by run_id, but allow multiple corpus entries per query
queries_dict = {}      # query_id (= run_id) -> {"_id": query_id, "text": query}
corpus_rows = []       # list of {"_id": corpus_id, "text": corpus}
qrels_rows = []        # list of (query-id, corpus-id, score)

for i in range(len(klg_df)):
    run_id = str(klg_df.iloc[i]["run_id"])
    knowledge = str(klg_df.iloc[i]["knowledge"])
    condition = str(klg_df.iloc[i]["condition"])

    positive_key = f"Condition: {condition}\nKnowledge: {knowledge}"

    # ---- load query from corresponding run ----
    msg_path = os.path.join(seimei_run_dir, run_id, "messages.json")
    if not os.path.exists(msg_path):
        # if messages file missing, skip
        continue

    with open(msg_path, "r", encoding="utf-8") as f:
        messages = json.load(f)
    
    query = None
    for message_dict in messages:
        if message_dict.get("role") == "user":
            query = message_dict.get("content", "")
            break

    if not query:
        # no user message found; skip this row
        continue

    # extract the actual query text (adjust this if your prompt format changes)
    split_token = "answer the question below:\n\n"
    if split_token in query:
        query = query.split(split_token, 1)[1]

    query_id = run_id
    corpus_id = i
    corpus = positive_key
    score = 1

    # store / dedupe query
    if query_id not in queries_dict:
        queries_dict[query_id] = {"_id": query_id, "text": query}

    # store corpus + qrels
    corpus_rows.append({"_id": str(corpus_id), "text": corpus})
    qrels_rows.append({"query-id": query_id, "corpus-id": corpus_id, "score": score})

# ---- save queries.jsonl ----
with open(query_save_path, "w", encoding="utf-8") as f:
    for q in queries_dict.values():
        json.dump(q, f, ensure_ascii=False)
        f.write("\n")

# ---- save corpus.jsonl ----
with open(corpus_save_path, "w", encoding="utf-8") as f:
    for c in corpus_rows:
        json.dump(c, f, ensure_ascii=False)
        f.write("\n")

# ---- save qrels/test.tsv ----
with open(qrel_save_path, "w", encoding="utf-8") as f:
    f.write("query-id\tcorpus-id\tscore\n")
    for row in qrels_rows:
        f.write(f"{row['query-id']}\t{row['corpus-id']}\t{row['score']}\n")


# Format conversion for an error

In [2]:
import json
import csv

knowledge_path = "../seimei_knowledge/exp4_train.csv"
seimei_run_dir = "../seimei_runs/"
query_save_path = "../exp4/train_klg_retrieval_eval/queries.jsonl"
corpus_save_path = "../exp4/train_klg_retrieval_eval/corpus.jsonl"
qrel_save_path = "../exp4/train_klg_retrieval_eval/qrels/test.tsv"

# Input files
queries_in = "../exp4/train_klg_retrieval_eval/queries.jsonl"
qrels_in = "../exp4/train_klg_retrieval_eval/qrels/test.tsv"

# Output files
queries_out = "../exp4/train_klg_retrieval_eval/queries_.jsonl"
qrels_out = "../exp4/train_klg_retrieval_eval/qrels/test_.tsv"

# 1. Build mapping from old run_id -> new numeric id (0,1,2,...) in order of appearance
id_map = {}
next_id = 0

with open(queries_in, "r", encoding="utf-8") as fin, \
     open(queries_out, "w", encoding="utf-8") as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue

        obj = json.loads(line)
        old_id = obj["_id"]

        if old_id not in id_map:
            id_map[old_id] = str(next_id)  # keep as string for consistency in TSV
            next_id += 1

        # Replace _id with new id
        obj["_id"] = id_map[old_id]

        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

# 2. Apply the same mapping to qrels.tsv (query-id column)
with open(qrels_in, "r", encoding="utf-8") as fin, \
     open(qrels_out, "w", encoding="utf-8", newline="") as fout:

    reader = csv.reader(fin, delimiter="\t")
    writer = csv.writer(fout, delimiter="\t")

    # Copy header as-is
    header = next(reader)
    writer.writerow(header)

    # Expected columns: query-id, corpus-id, score
    for row in reader:
        if not row:
            continue

        old_qid = row[0]

        if old_qid not in id_map:
            raise KeyError(f"Query ID '{old_qid}' in qrels not found in queries.jsonl mapping")

        row[0] = id_map[old_qid]
        writer.writerow(row)

print("Done. Written:", queries_out, "and", qrels_out)


Done. Written: ../exp4/train_klg_retrieval_eval/queries_.jsonl and ../exp4/train_klg_retrieval_eval/qrels/test_.tsv
